In [20]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
print(model.encode(["test sentence"]).shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(1, 384)


Imports and loading the data

In [21]:
import re
import numpy as np
import pandas as pd

df = pd.read_csv("../data/raw/Reviews.csv")
print("Raw shape:", df.shape)
df.head(3)

Raw shape: (568454, 10)


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...


Removing duplicates and bad rows

In [22]:
n_raw = len(df)

# The same review is often attached to several product variants (flavours/sizes).
# A review is identified by who wrote it, when, and what it says.
df = df.drop_duplicates(subset=["UserId", "ProfileName", "Time", "Text"])
print(f"After de-duplication: {len(df):,} rows ({1 - len(df)/n_raw:.1%} were duplicates)")

# Impossible rows: more 'helpful' votes than total votes
df = df[df.HelpfulnessNumerator <= df.HelpfulnessDenominator]

# Missing text
df = df.dropna(subset=["Text"])
df["Summary"] = df["Summary"].fillna("")
print(f"After cleaning bad rows: {len(df):,} rows")

After de-duplication: 393,933 rows (30.7% were duplicates)
After cleaning bad rows: 393,931 rows


Cleaning the text

In [23]:
HTML_TAG = re.compile(r"<[^>]+>")
WHITESPACE = re.compile(r"\s+")

def clean_text(s: str) -> str:
    s = HTML_TAG.sub(" ", str(s))                       # remove <br /> etc.
    s = s.replace("&quot;", '"').replace("&amp;", "&").replace("&#34;", '"')
    return WHITESPACE.sub(" ", s).strip()               # collapse spaces

df["Text"] = df["Text"].map(clean_text)
df["Summary"] = df["Summary"].map(clean_text)

# Quick check: no HTML left
print("Rows still containing '<br':", df.Text.str.contains("<br", regex=False).sum())

Rows still containing '<br': 0


Adding helper columns

In [24]:
df["date"] = pd.to_datetime(df["Time"], unit="s")
df["year"] = df["date"].dt.year

df["sentiment"] = np.select(
    [df.Score <= 2, df.Score == 3],
    ["negative", "neutral"],
    default="positive",
)

# Summary acts like a headline; it gives the embedding model extra signal
df["text_for_embedding"] = df["Summary"] + ". " + df["Text"]

df = df.reset_index(drop=True)
df[["Score", "sentiment", "year", "text_for_embedding"]].head()

,Score,sentiment,year,text_for_embedding
0,5,positive,2011,Good Quality Dog Food. I have bought several o...
1,1,negative,2012,Not as Advertised. Product arrived labeled as ...
2,4,positive,2008,"""Delight"" says it all. This is a confection th..."
3,2,negative,2011,Cough Medicine. If you are looking for the sec...
4,5,positive,2012,Great taffy. Great taffy at a great price. The...


Taking a stratified sample of 50K

In [25]:
def stratified_sample(data: pd.DataFrame, n: int = 50_000, seed: int = 42) -> pd.DataFrame:
    """Sample proportionally within each (Score, year) group so the mix stays representative."""
    frac = n / len(data)
    parts = []
    for _, group in data.groupby(["Score", "year"]):
        k = int(round(len(group) * frac))
        if k > 0:
            parts.append(group.sample(n=min(k, len(group)), random_state=seed))
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

sample = stratified_sample(df)
print("Sample size:", len(sample))

# Check: star mix should match the full data
check = pd.DataFrame({
    "full": df.Score.value_counts(normalize=True),
    "sample": sample.Score.value_counts(normalize=True),
}).sort_index().round(3)
check

Sample size: 50001


,full,sample
Score,,
1,0.092,0.092
2,0.053,0.053
3,0.076,0.076
4,0.142,0.142
5,0.637,0.637


Save the sample

In [26]:
sample.to_pickle("../artifacts/sample.pkl")
print("Saved ../artifacts/sample.pkl")

Saved ../artifacts/sample.pkl


Quick charts for write-up

In [4]:
import pandas as pd
import plotly.express as px

sample = pd.read_pickle("../artifacts/sample.pkl")

yearly = sample.groupby("year").size().reset_index(name="reviews")
px.bar(yearly, x="year", y="reviews", title="Reviews per year (sample)").show()

sample["n_words"] = sample.Text.str.split().str.len()
px.histogram(sample[sample.n_words < 600], x="n_words", nbins=60,
             title="Review length (words)").show()